# Automatización WB — Fase 3 (Búsqueda de vuelos) — asignar Instructor + Actividad

Equivalente al notebook NB (`Automatizacion_Fase3_Asignacion_Instructor_Vuelos.ipynb`), pero
para las flotas WB **B767** y **B787**. Son **dos procesos separados**: corre este notebook una
vez con `FLOTA_WB = "767"` y, por separado, otra vez con `FLOTA_WB = "787"`.

Continúa donde quedó `Automatizacion_WB_Fase1_2_Demanda_Instructores.ipynb`, que reservó
`"LCK B767"` / `"LCK B787"` en la Matriz para ciertos (instructor, fecha).

**Diferencia estructural clave con NB** (no inventada, viene de `generar_reporte_pairings.py`,
ya validado contra el roster real de referencia): en WB **un "bloque" = 1 solo pairing**
(ida + vuelta), no 2 pairings combinados como en NB. Un pairing LIM-SCL-LIM vuelve el mismo
día (como NB); un pairing LIM-MIA-LIM vuelve un día distinto (viaje de varios días).

**Punto abierto de la Fase 1/2, resuelto acá:** cuando se encuentra el pairing real que cubre
una reserva de la Matriz, si ese pairing es LIM-MIA-LIM, la fecha de vuelta real (distinta a la
de ida) queda calculada y reportada en este notebook — pero la escritura de esa segunda celda
en la Matriz se hace en el notebook de Fase 4 (que sí escribe en Sheets), no acá.

**No modifica** `generar_reporte_pairings.py` (sigue intacto); se copian sus funciones
(`cargar_y_filtrar`, la lógica de bloques) adaptadas para leer directo desde BigQuery en vez de
un CSV, y ya con `create_bqstorage_client=False` + los fixes de formato Excel encontrados en la
sesión de automatización NB (STD/STA/HBT como texto).


## 1. Instalar dependencias y autenticar

In [ ]:
!pip install -q gspread openpyxl


In [ ]:
import re
import unicodedata
import collections
from collections import defaultdict
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
import gspread
from google.auth import default
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.comments import Comment
from openpyxl.worksheet.datavalidation import DataValidation

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
client = bigquery.Client(project="datadem-home")  # billing project (ver notas NB: 403 en operations-data-prod)
print("Autenticado: BigQuery + Google Sheets con tu cuenta.")


## 2. Elegir la flota WB

Igual que en el notebook de Fase 1/2: cambia `FLOTA_WB` y vuelve a correr todo el notebook
para la otra flota.

In [ ]:
FLOTA_WB = "767"  # o "787"

WB_CONFIG = {
    "767": {
        "subfleets": {"763"},
        "actividad_matriz": "LCK B767",
    },
    "787": {
        "subfleets": {"788", "789"},
        "actividad_matriz": "LCK B787",
    },
}
if FLOTA_WB not in WB_CONFIG:
    raise ValueError("FLOTA_WB debe ser '767' o '787'")
cfg = WB_CONFIG[FLOTA_WB]
print(f"Flota seleccionada: WB {FLOTA_WB} -> actividad '{cfg['actividad_matriz']}', subflotas BigQuery {cfg['subfleets']}")


## 3. Query a BigQuery (ambas subflotas WB) + filtrado de candidatos

Query y reglas de `generar_reporte_pairings.py` (validado contra "Pairings WB SEPTIEMBRE-2026 -
Vuelos", ver docstring del script original): rutas LIM-MIA/MIA-LIM y LIM-SCL/SCL-LIM, solo
lunes a viernes, presentación no domingo, `dia_duty` contiguo y con rango <= 2, y solo los
números de vuelo de entrenamiento (`ALLOWED_FLIGHTS`).

**Corrección aplicada acá:** el archivo `consulta BQ WB.sql` tenía `subfleet_code IN ('762',
'788', '789')` — `'762'` es un typo, el código real de B767 es **`'763'`** (confirmado por los
propios comentarios de `generar_reporte_pairings.py`, que documentan "B767 (subfleet 763)").
Se corrige en la query de abajo.

In [ ]:
MES_OBJETIVO = 10
ANIO_OBJETIVO = 2026

QUERY_WB = """
SELECT
  pairing_id                       AS trip,
  pairing_start_date               AS fecha_inicio_trip,
  flight_start_date_local_time     AS inicio_vuelo_lt,
  flight_month_description         AS mes,
  duty_calendar_day_number         AS dia_duty,
  flight_number                    AS vuelo,
  departure_airport_code           AS dep,
  arrival_airport_code             AS arr,
  flight_departure_time_crew_base  AS std_hb,
  flight_arrival_hour_block_time   AS sta_hb,
  flight_block_time                AS hbt,
  subfleet_code                    AS sub_fleet,
  is_crew_passenger                AS pax,
  duty_presentation_date_at        AS presentacion_duty_date_lt

FROM `operations-data-prod.carmen_gold.crew_pairing_carmen_system`

WHERE
  flight_start_date_local_time BETWEEN DATE '2026-09-01' AND DATE '2026-10-31'
  AND subsidiary_code IN ('LP')
  AND load_type_code = 'FP'
  AND crew_range_type_code = 'SAB'
  AND subfleet_code IN ('763', '788', '789')

QUALIFY
  CASE
    WHEN load_type_code = 'FP' AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'FP' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    WHEN load_type_code = 'ES' AND
         MAX(CASE WHEN load_type_code = 'FP' THEN 0 ELSE 0 END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year) = -1 AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'ES' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    ELSE -1
  END = 0

ORDER BY pairing_id ASC
"""

df_raw = client.query(QUERY_WB).to_dataframe(create_bqstorage_client=False)
print(f"Filas descargadas (B767+B787): {len(df_raw)}")


## 4. `cargar_y_filtrar` — copia adaptada de `generar_reporte_pairings.py`

Misma lógica de filtrado exacta (reglas 1-5 del docstring original), adaptada para leer
directo del DataFrame de BigQuery (fechas ISO, `pd.to_datetime`) en vez del CSV con fechas en
texto español que usaba el script original.

In [ ]:
ALLOWED_ROUTES = {
    ("LIM", "MIA"), ("MIA", "LIM"),
    ("LIM", "SCL"), ("SCL", "LIM"),
}
ALLOWED_FLIGHTS = {2480, 2481, 2695, 2694, 2698, 2699, 2413, 2412, 2697, 2696}
DIAS_OK = {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday"}
WD_ES = {
    "Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles",
    "Thursday": "jueves", "Friday": "viernes", "Saturday": "sábado",
    "Sunday": "domingo",
}
HEADERS_WB = ["Pairing ID", "FECHA REAL", "MES", "Day of Week", "Flight No",
              "Dep Stn", "Arr Stn", "STD", "STA", "DAY", "AC Type", "HBT", "DIA_DUTY"]


def cargar_y_filtrar_wb(df_raw: pd.DataFrame, mes_objetivo: int, anio_objetivo: int) -> pd.DataFrame:
    df = df_raw.copy()
    df["vuelo"] = df["vuelo"].astype(int)
    df["dia_duty"] = df["dia_duty"].astype(int)

    df["inicio_vuelo_lt_dt"] = pd.to_datetime(df["inicio_vuelo_lt"])
    df["presentacion_duty_date_lt_dt"] = pd.to_datetime(df["presentacion_duty_date_lt"])
    df["wd_vuelo"] = df["inicio_vuelo_lt_dt"].dt.day_name()
    df["wd_pres"] = df["presentacion_duty_date_lt_dt"].dt.day_name()
    df["dia_semana"] = df["wd_vuelo"].map(WD_ES)

    df["_pk"] = df["trip"].astype(str) + "|" + df["fecha_inicio_trip"].astype(str)

    df["leg_dia_ok"] = df["wd_vuelo"].isin(DIAS_OK)
    df["leg_pres_ok"] = df["wd_pres"] != "Sunday"
    df["leg_ruta_ok"] = list(zip(df["dep"], df["arr"]))
    df["leg_ruta_ok"] = df["leg_ruta_ok"].isin(ALLOWED_ROUTES) & df["vuelo"].isin(ALLOWED_FLIGHTS)

    g = df.groupby("_pk")
    trip_dia_ok = g["leg_dia_ok"].transform("all")
    trip_pres_ok = g["leg_pres_ok"].transform("all")

    idx_primera_pierna = g["dia_duty"].idxmin()
    primeras_piernas = df.loc[idx_primera_pierna, ["_pk", "inicio_vuelo_lt_dt"]]
    primeras_piernas["mes_ok"] = (primeras_piernas["inicio_vuelo_lt_dt"].dt.month == mes_objetivo) & \
                                  (primeras_piernas["inicio_vuelo_lt_dt"].dt.year == anio_objetivo)
    mapa_mes_ok = primeras_piernas.set_index("_pk")["mes_ok"]
    trip_mes_ok = df["_pk"].map(mapa_mes_ok)
    trip_ruta_ok = g["leg_ruta_ok"].transform("all")

    def dia_duty_contiguo_y_corto(s):
        vals = sorted(s.dropna().unique())
        if not vals:
            return False
        rango_ok = (vals[-1] - vals[0]) <= 2
        contiguo = all(b - a == 1 for a, b in zip(vals, vals[1:]))
        return rango_ok and contiguo

    trip_dia_duty_ok = g["dia_duty"].transform(dia_duty_contiguo_y_corto)

    df["trip_valido"] = trip_dia_ok & trip_mes_ok & trip_pres_ok & trip_ruta_ok & trip_dia_duty_ok

    validos = df[df["trip_valido"]].copy()
    validos["fecha_real_fmt"] = validos["inicio_vuelo_lt_dt"].dt.strftime("%d/%m/%Y")

    validos = validos.rename(columns={
        "trip": "Pairing ID", "fecha_real_fmt": "FECHA REAL", "mes": "MES",
        "dia_semana": "Day of Week", "vuelo": "Flight No", "dep": "Dep Stn", "arr": "Arr Stn",
        "std_hb": "STD", "sta_hb": "STA", "pax": "DAY", "sub_fleet": "AC Type",
        "hbt": "HBT", "dia_duty": "DIA_DUTY",
    })

    validos = validos.sort_values(["_pk", "DIA_DUTY"])[HEADERS_WB + ["_pk"]]

    # Un pairing con reglas OK debería tener sus 2 piernas completas (ida+vuelta); si BigQuery
    # solo trae 1 (ej. el reload de un mes se queda con la ida y pierde la vuelta que cae en el
    # mes siguiente con otro reload -> caso real ya encontrado con armar_tabla_wb.py), no se
    # puede armar el bloque -> se descarta y se reporta, no se inventa la pierna faltante.
    piernas_por_pk = validos.groupby("_pk").size()
    pk_incompletos = piernas_por_pk[piernas_por_pk < 2].index
    incompletos = validos[validos["_pk"].isin(pk_incompletos)].copy()
    validos = validos[~validos["_pk"].isin(pk_incompletos)].copy()

    return validos, incompletos


validos_todas_flotas, incompletos = cargar_y_filtrar_wb(df_raw, MES_OBJETIVO, ANIO_OBJETIVO)
validos = validos_todas_flotas[validos_todas_flotas["AC Type"].astype(str).isin(cfg["subfleets"])].copy()

print(f"Pairings válidos (ambas flotas WB): {validos_todas_flotas['_pk'].nunique()}")
print(f"Pairings incompletos descartados (1 sola pierna, ambas flotas): {incompletos['_pk'].nunique()}")
print(f"Pairings válidos de la flota elegida ({FLOTA_WB}): {validos['_pk'].nunique()}")
print(f"Filas de vuelo de la flota elegida: {len(validos)}")


## 5. Leer la Matriz real: qué (instructor, fecha) quedaron reservados

Misma Matriz, misma pestaña que NB y que la Fase 1/2 de WB. Busca la actividad de la flota
elegida (`"LCK B767"` o `"LCK B787"`).

In [ ]:
URL_MATRIZ = "https://docs.google.com/spreadsheets/d/19WmwaoLDZnArNztu_dJNwi7bjrGq-_cx_gw0aN96zfk/edit?gid=580414308"

sh_matriz = gc.open_by_url(URL_MATRIZ)
ws_matriz = sh_matriz.get_worksheet_by_id(580414308)

FILA_ENCABEZADO_FECHAS = 2
FILA_PRIMER_INSTRUCTOR = 3
COL_PRIMERA_FECHA = 3

valores_m = ws_matriz.get_all_values()

fila_fechas = valores_m[FILA_ENCABEZADO_FECHAS - 1]
fechas_matriz = [c.strip() if c.strip() else None for c in fila_fechas[COL_PRIMERA_FECHA - 1:]]

reservas = []  # (bp, nombre_matriz, fecha_str "dd/mm/yyyy")
for fila in valores_m[FILA_PRIMER_INSTRUCTOR - 1:]:
    if not fila or not fila[0].strip():
        continue
    bp = fila[0].strip().lstrip("'")
    nombre = fila[1].strip() if len(fila) > 1 else ""
    for j, celda in enumerate(fila[COL_PRIMERA_FECHA - 1:]):
        if celda.strip().upper() == cfg["actividad_matriz"].upper():
            fecha_str = fechas_matriz[j] if j < len(fechas_matriz) else None
            if fecha_str:
                reservas.append((bp, nombre, fecha_str))

print(f"Slots '{cfg['actividad_matriz']}' encontrados en la Matriz: {len(reservas)}")


## 6. Catálogo de instructores de esta flota (leído en vivo del Rol de Instructores)

A diferencia de NB (catálogo fijo con nombre legal completo, dado directamente por Fernando),
para WB no tenemos todavía los nombres legales completos de cada instructor -> se lee el
catálogo BP+Nombre en vivo desde el mismo Rol de Instructores que usa la Fase 1/2 (columna
`IDE B767`/`IDE B787` = OK), en vez de inventar un nombre legal. Si más adelante Fernando
confirma los nombres legales reales, se puede agregar una tercera columna igual que en NB.

In [ ]:
URL_ROL_INSTRUCTORES = "https://docs.google.com/spreadsheets/d/1yMvgb_O4qxpCCE4bAqD4XhW2GQI9ZObf2EAnymXaReE/edit?gid=1933640306"
ROL_COL_IDE = {"767": "IDE B767", "787": "IDE B787"}[FLOTA_WB]

sh_rol_ins = gc.open_by_url(URL_ROL_INSTRUCTORES)
ws_rol_ins = sh_rol_ins.get_worksheet_by_id(1933640306)

registros_rol = ws_rol_ins.get_all_records()
df_rol = pd.DataFrame(registros_rol)
if ROL_COL_IDE not in df_rol.columns:
    raise RuntimeError(f"No encontré la columna '{ROL_COL_IDE}' en Rol Instructores -> columnas: {list(df_rol.columns)}")

instructores_ide_df = df_rol[df_rol[ROL_COL_IDE].astype(str).str.strip().str.upper() == "OK"].copy()
INSTRUCTORES_DATA_WB = list(zip(instructores_ide_df["Nombre"], instructores_ide_df.iloc[:, 0].astype(str)))

print(f"Instructores {ROL_COL_IDE} = OK (catálogo para esta corrida): {len(INSTRUCTORES_DATA_WB)}")
for nombre, bp in INSTRUCTORES_DATA_WB:
    print(f"  {bp}  {nombre}")


## 7. Emparejar cada reserva de la Matriz con un pairing real de esa fecha

Reglas (mismo principio que NB, adaptado a que acá "bloque" = 1 pairing, no 2):
- Se busca un `_pk` cuya PRIMERA pierna (fecha de ida) caiga en la fecha EXACTA reservada.
- Si no hay ningún pairing válido esa fecha, se reporta en `sin_bloque` — no se busca fecha
  alternativa.
- El nombre se matchea normalizando tildes/mayúsculas contra `INSTRUCTORES_DATA_WB`; si no
  matchea, se reporta en `sin_match_nombre`.
- Si el pairing encontrado es LIM-MIA-LIM (ida y vuelta en fechas distintas), se calcula y
  reporta la fecha real de vuelta — la segunda celda de la Matriz para esa fecha se escribe en
  el notebook de Fase 4, no acá (este notebook no escribe en Sheets).

In [ ]:
def normalizar_nombre(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return s.strip().lower()


def normalizar_fecha(fecha_str):
    d, m, a = fecha_str.strip().split("/")
    return (int(d), int(m), int(a))


def armar_pairings_dict(validos: pd.DataFrame):
    """_pk -> (fila_ida, fila_vuelta) ordenadas por DIA_DUTY, cada una una Serie de `validos`."""
    pairings = {}
    for pk, grupo in validos.groupby("_pk", sort=False):
        filas = [row for _, row in grupo.sort_values("DIA_DUTY").iterrows()]
        if len(filas) != 2:
            continue  # ya filtrado antes, pero por seguridad no asumir
        pairings[pk] = (filas[0], filas[1])
    return pairings


pairings_por_pk = armar_pairings_dict(validos)
pairings_por_fecha_ida = defaultdict(list)
for pk, (ida, _vta) in pairings_por_pk.items():
    pairings_por_fecha_ida[normalizar_fecha(ida["FECHA REAL"])].append(pk)


def emparejar_matriz_con_pairings(reservas, pairings_por_pk, pairings_por_fecha_ida):
    mapa_nombre_canonico = {normalizar_nombre(n): n for n, _bp in INSTRUCTORES_DATA_WB}

    usados_pk = set()
    asignaciones_por_pk = {}   # pk -> (nombre_instructor, actividad, fecha_vuelta_real_o_None)
    sin_bloque = []
    sin_match_nombre = []

    for bp, nombre_matriz, fecha_str in reservas:
        nombre_canonico = mapa_nombre_canonico.get(normalizar_nombre(nombre_matriz))
        if nombre_canonico is None:
            sin_match_nombre.append((bp, nombre_matriz, fecha_str))
            continue

        fecha_norm = normalizar_fecha(fecha_str)
        candidatos = [pk for pk in pairings_por_fecha_ida.get(fecha_norm, []) if pk not in usados_pk]

        if not candidatos:
            sin_bloque.append((bp, nombre_canonico, fecha_str))
            continue

        pk_elegido = candidatos[0]
        usados_pk.add(pk_elegido)
        ida, vta = pairings_por_pk[pk_elegido]
        fecha_vuelta_real = vta["FECHA REAL"]
        fecha_vuelta_distinta = normalizar_fecha(fecha_vuelta_real) != fecha_norm
        asignaciones_por_pk[pk_elegido] = (
            nombre_canonico, cfg["actividad_matriz"],
            fecha_vuelta_real if fecha_vuelta_distinta else None,
        )

    return asignaciones_por_pk, sin_bloque, sin_match_nombre


asignaciones_por_pk, sin_bloque, sin_match_nombre = emparejar_matriz_con_pairings(
    reservas, pairings_por_pk, pairings_por_fecha_ida)

n_mia_multi_dia = sum(1 for _n, _a, fv in asignaciones_por_pk.values() if fv is not None)

print(f"Reservas de la Matriz ({cfg['actividad_matriz']}): {len(reservas)}")
print(f"Asignadas a un pairing real: {len(asignaciones_por_pk)}")
print(f"  de las cuales con vuelta en fecha DISTINTA a la ida (LIM-MIA-LIM, revisar Fase 4): {n_mia_multi_dia}")
print(f"Sin pairing disponible esa fecha exacta (revisar a mano): {len(sin_bloque)}")
for bp, nombre, fecha in sin_bloque:
    print(f"  - {nombre} (BP {bp}) reservado el {fecha}, no hay pairing válido ese día")
print(f"Nombres sin match en el catálogo (revisar a mano): {len(sin_match_nombre)}")
for bp, nombre, fecha in sin_match_nombre:
    print(f"  - '{nombre}' (BP {bp}, {fecha}) no matchea ningún instructor {ROL_COL_IDE}")

if n_mia_multi_dia:
    print("\nPairings con fecha de vuelta distinta a la de ida (2da celda de Matriz pendiente en Fase 4):")
    for pk, (nombre, _act, fecha_vuelta) in asignaciones_por_pk.items():
        if fecha_vuelta is not None:
            ida, _vta = pairings_por_pk[pk]
            print(f"  - {nombre}: ida {ida['FECHA REAL']} -> vuelta {fecha_vuelta} (pairing {pk})")


## 8. Armar el Excel de pairings, ya con Instructor + Actividad completados

Adaptado de `construir_reporte_bloques` (`generar_reporte_pairings.py`), con 3 agregados:
1. Parámetro `asignaciones_por_pk` (dict `{_pk: (instructor, actividad, fecha_vuelta_o_None)}`)
   que llena Instructor (columna O) y Actividad (columna Q) directamente, en vez de dejarlos
   en blanco para llenar a mano.
2. STD/STA/HBT forzados a texto (`str(...)` + `number_format="@"`) — mismo fix ya encontrado
   en la sesión NB para evitar `#¡VALOR!` en las fórmulas de la columna Q.
3. Hoja "Resumen Final" con CUPOS por flota (B767=5, B787=6, manual "Dotación por flota"),
   igual patrón que NB.

In [ ]:
CUPOS_POR_FLOTA_WB = {"767": 5, "787": 6}
CUPOS_ESTA_FLOTA = CUPOS_POR_FLOTA_WB[FLOTA_WB]

TITULO_1 = "Comenzar asignando vuelos los Jueves y Viernes (porque el SAB es DO)"
TITULO_2 = "LCK de ida (porque va OP) y de retorno DGAC o RECA"
HEADERS_BLOQUE = ["Pairing ID", "FECHA REAL", "MES", "Day of Week", "Flight No",
                  "Dep Stn", "Arr Stn", "STD", "STA", "DAY", "AC Type", "HBT", "DIA_DUTY"]
COLUMNAS_HORA_WB = {"STD", "STA"}


def construir_bloques_wb(pairings_por_pk, out_path, asignaciones_por_pk=None, blank_rows=2):
    asignaciones_por_pk = asignaciones_por_pk or {}
    wb = Workbook()
    ws = wb.active
    ws.title = "Pairings"

    bold = Font(bold=True)
    header_fill = PatternFill("solid", fgColor="FFFF00")
    dutyid_fill = PatternFill("solid", fgColor="FFC000")
    resumen_fill = PatternFill("solid", fgColor="D9E1F2")
    thin = Side(style="thin", color="999999")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal="center")

    COL_O, COL_P, COL_Q = 15, 16, 17
    COL_RESUMEN_INI = 18
    RESUMEN_HEADERS = ["Pairing ID", "Fecha", "Día Sem", "Vuelo", "Ruta",
                        "Instructor", "Actividad", "Inicio", "Fin"]

    ws.cell(row=1, column=2, value=TITULO_1).font = bold
    ws.cell(row=2, column=2, value=TITULO_2).font = bold

    for j, h in enumerate(RESUMEN_HEADERS):
        c = ws.cell(row=3, column=COL_RESUMEN_INI + j, value=h)
        c.font = bold
        c.fill = resumen_fill
        c.border = border
        c.alignment = center

    dv_instructor = None
    if INSTRUCTORES_DATA_WB:
        wi = wb.create_sheet("Instructores")
        for j, h in enumerate(["Instructor", "BP"]):
            c = wi.cell(row=1, column=1 + j, value=h)
            c.font = bold
            c.border = border
        for i, (nombre, bp) in enumerate(INSTRUCTORES_DATA_WB, start=2):
            wi.cell(row=i, column=1, value=nombre).border = border
            wi.cell(row=i, column=2, value=bp).border = border
        for c, w in zip("AB", (22, 12)):
            wi.column_dimensions[c].width = w
        dv_instructor = DataValidation(
            type="list", formula1=f"=Instructores!$A$2:$A${1 + len(INSTRUCTORES_DATA_WB)}",
            allow_blank=True)
        ws.add_data_validation(dv_instructor)

    fila = 5
    resumen_final_rows = []  # (pk, ida_row_dict, vta_row_dict, r_header, cupos)

    for pk, (ida, vta) in pairings_por_pk.items():
        pre_row, header_row = fila - 1, fila
        leg1_row, leg2_row = fila + 1, fila + 2
        post_row_1, post_row_2 = fila + 3, fila + 4

        for j, h in enumerate(HEADERS_BLOQUE):
            c = ws.cell(row=header_row, column=2 + j, value=h)
            c.font = bold
            c.border = border
            c.alignment = center
            if h == "MES":
                c.fill = header_fill
            if h == "DIA_DUTY":
                c.fill = dutyid_fill

        asignacion = asignaciones_por_pk.get(pk)

        c_ins = ws.cell(row=header_row, column=COL_O)
        c_ins.fill = dutyid_fill
        c_ins.font = bold
        c_ins.comment = Comment("Elegir el instructor de la lista", "Automatizacion_WB_Fase3")
        if dv_instructor is not None:
            dv_instructor.add(c_ins)
        if asignacion is not None:
            c_ins.value = asignacion[0]

        c_act = ws.cell(row=pre_row, column=COL_Q)
        c_act.fill = dutyid_fill
        c_act.font = bold
        c_act.comment = Comment(
            "Tipo de actividad (ej. LCK B767, Reentrenamiento B767, CHEQUEO BI ANUAL INST B767)",
            "Automatizacion_WB_Fase3")
        if asignacion is not None:
            c_act.value = asignacion[1]

        for r, pierna in zip((leg1_row, leg2_row), (ida, vta)):
            for j, h in enumerate(HEADERS_BLOQUE):
                valor = pierna[h]
                if h in COLUMNAS_HORA_WB or h == "HBT":
                    valor = str(valor)
                c = ws.cell(row=r, column=2 + j, value=valor)
                c.border = border
                c.alignment = center
                if h in COLUMNAS_HORA_WB:
                    c.number_format = "@"
            c_hbt = ws.cell(row=r, column=2 + HEADERS_BLOQUE.index("HBT"))
            c_hbt.number_format = "@"

        ws.cell(row=leg2_row, column=COL_O, value=f'=IF(O{header_row}="","",O{header_row})')

        ws.cell(row=header_row, column=COL_Q, value=f'="- "&G{leg1_row}&"-"&H{leg1_row}')
        ws.cell(row=leg1_row, column=COL_Q,
                value=(f'="- LA "&F{leg1_row}&" ("&LEFT(I{leg1_row},5)&"-"'
                       f'&LEFT(J{leg1_row},5)&" hrs)"'))
        ws.cell(row=leg2_row, column=COL_Q, value=f'=IF(Q{pre_row}="","",Q{pre_row})')
        ws.cell(row=post_row_1, column=COL_Q, value=f'="- "&G{leg2_row}&"-"&H{leg2_row}')
        ws.cell(row=post_row_2, column=COL_Q,
                value=(f'="- LA "&F{leg2_row}&" ("&LEFT(I{leg2_row},5)&"-"'
                       f'&LEFT(J{leg2_row},5)&" hrs)"'))

        resumen_valores = [
            f"=B{leg1_row}", f"=C{leg1_row}", f"=E{leg1_row}",
            f'=F{leg1_row}&"/"&F{leg2_row}',
            f'=G{leg1_row}&"-"&H{leg1_row}&"-"&G{leg1_row}',
            f'=IF(O{header_row}="","",O{header_row})',
            f'=IF(Q{pre_row}="","",Q{pre_row})',
            f"=C{leg1_row}", f"=C{leg2_row}",
        ]
        for j, val in enumerate(resumen_valores):
            c = ws.cell(row=header_row, column=COL_RESUMEN_INI + j, value=val)
            c.border = border
            c.alignment = center

        resumen_final_rows.append((pk, ida, vta, asignacion, CUPOS_ESTA_FLOTA))
        fila = post_row_2 + 1 + blank_rows

    anchos = [10, 12, 10, 12, 10, 9, 9, 10, 10, 6, 9, 9, 10]
    for j, w in enumerate(anchos):
        ws.column_dimensions[ws.cell(row=1, column=2 + j).column_letter].width = w
    for col, w in ((COL_O, 20), (COL_P, 3), (COL_Q, 30)):
        ws.column_dimensions[ws.cell(row=1, column=col).column_letter].width = w
    for j, w in enumerate([10, 12, 10, 12, 16, 20, 26, 12, 12]):
        ws.column_dimensions[ws.cell(row=1, column=COL_RESUMEN_INI + j).column_letter].width = w
    ws.freeze_panes = "B4"

    wr = wb.create_sheet("Resumen Final")
    resumen_final_headers = ["Pairing ID", "Fecha Ida", "Fecha Vuelta", "DíaSEM", "Vuelo",
                              "Ruta", "BP INS", "INS", "ACTIVIDAD", "FLOTA", "CUPOS"]
    for j, h in enumerate(resumen_final_headers):
        c = wr.cell(row=1, column=1 + j, value=h)
        c.font = bold
        c.border = border

    mapa_bp = {nombre: bp for nombre, bp in INSTRUCTORES_DATA_WB}
    for i, (pk, ida, vta, asignacion, cupos) in enumerate(resumen_final_rows, start=2):
        nombre_ins = asignacion[0] if asignacion else ""
        actividad = asignacion[1] if asignacion else ""
        valores = [
            pk, ida["FECHA REAL"], vta["FECHA REAL"], ida["Day of Week"],
            f'{ida["Flight No"]}/{vta["Flight No"]}',
            f'{ida["Dep Stn"]}-{ida["Arr Stn"]}-{ida["Dep Stn"]}',
            mapa_bp.get(nombre_ins, ""), nombre_ins, actividad, ida["AC Type"], cupos,
        ]
        for j, val in enumerate(valores):
            c = wr.cell(row=i, column=1 + j, value=val)
            c.border = border

    for col, w in zip("ABCDEFGHIJK", (12, 12, 12, 10, 12, 16, 9, 20, 20, 8, 8)):
        wr.column_dimensions[col].width = w
    wr.freeze_panes = "A2"
    wr.auto_filter.ref = wr.dimensions

    wb.save(out_path)


OUT = f"Pairings_WB_{FLOTA_WB}_con_instructores.xlsx"
construir_bloques_wb(pairings_por_pk, OUT, asignaciones_por_pk=asignaciones_por_pk)
print(f"Archivo generado: {OUT}")
print(f"Pairings con Instructor+Actividad completados: {len(asignaciones_por_pk)} de {len(pairings_por_pk)} pairings totales")


## 9. QA: recalcular en Python que el llenado quedó bien

In [ ]:
nombres_catalogo = {n for n, _bp in INSTRUCTORES_DATA_WB}
malos = [(pk, n) for pk, (n, _a, _fv) in asignaciones_por_pk.items() if n not in nombres_catalogo]
print("Asignaciones con nombre fuera del catálogo (deben ser 0):", len(malos))

print("Pairings asignados (deben ser todos únicos, por construcción del dict):", len(asignaciones_por_pk))

inconsistencias_fecha = 0
for pk, (_n, _a, _fv) in asignaciones_por_pk.items():
    ida, _vta = pairings_por_pk[pk]
    fecha_pairing = ida["FECHA REAL"]
    if not any(normalizar_fecha(f) == normalizar_fecha(fecha_pairing) for _bp, _nom, f in reservas):
        inconsistencias_fecha += 1
print("Pairings asignados cuya fecha de ida no estaba reservada en la Matriz (deben ser 0):", inconsistencias_fecha)

ids_usados = list(pairings_por_pk.keys())
rep = [pid for pid, c in collections.Counter(ids_usados).items() if c > 1]
print("_pk repetidos entre pairings (deben ser 0):", len(rep))

fuera_de_flota = validos[~validos["AC Type"].astype(str).isin(cfg["subfleets"])]
print(f"Filas coladas de otra subflota (deben ser 0): {len(fuera_de_flota)}")


## 10. Descargar el archivo final

In [ ]:
from google.colab import files
files.download(OUT)


## 11. Estado y pendientes — sin inventar nada

### Lo que se automatiza en este notebook
- Query a BigQuery para ambas subflotas WB (763, 788, 789) — corrige el typo `'762'`→`'763'`
  que tenía `consulta BQ WB.sql`.
- Filtra candidatos con las reglas ya validadas de `generar_reporte_pairings.py` (rutas, número
  de vuelo, día de semana, presentación, `dia_duty` contiguo ≤2), separadas por subflota según
  `FLOTA_WB`.
- Lee la Matriz real y detecta los slots `"LCK B767"`/`"LCK B787"` reservados por la Fase 1/2.
- Empareja cada reserva con el pairing real de esa fecha exacta de ida, y llena
  Instructor + Actividad en el Excel (hoja "Pairings" + "Resumen Final", con CUPOS por flota).
- Reporta aparte (sin inventar) los casos sin pairing disponible esa fecha, los nombres sin
  match, y los pairings LIM-MIA-LIM cuya fecha de vuelta es distinta de la de ida.

### Confirmado con Fernando (2026-09-24)
- Cupos por vuelo: B767 = 5, B787 = 6 (manual, "Dotación por flota").
- El typo `'762'` en `consulta BQ WB.sql` es un error (debe ser `'763'`) — corregido acá.

### Supuesto pendiente de confirmar
- El catálogo de instructores WB (`INSTRUCTORES_DATA_WB`) se lee en vivo del Rol de
  Instructores (BP + Nombre), NO es una lista fija dada por Fernando como en NB (ahí no
  tenemos los nombres legales completos todavía) — si hace falta el nombre legal completo para
  algún cruce futuro, hay que pedirlo y agregarlo como columna aparte.

### Lo que sigue sin automatizar / pendiente para la Fase 4 de WB
- **La segunda celda de la Matriz para pairings LIM-MIA-LIM** (fecha de vuelta, distinta a la
  de ida) — este notebook la calcula y la reporta, pero la escritura real en Sheets se hace en
  `Automatizacion_WB_Fase4_Consolidacion.ipynb`.
- Otras actividades (auditorías, habilitación, reentrenamiento, etc.) — 100% manuales.
- PDR / descanso reglamentario exacto contra otros vuelos de línea del instructor — no se
  calcula, igual que en NB.
- Si `sin_bloque` o `sin_match_nombre` no están vacíos, requieren ajuste manual antes de dar el
  archivo por definitivo.
